# 06 — RecBole: SASRec (sequential neural baseline)

Trains **SASRec** (transformer, sequential) and scores it through our shared
`evaluate_predictions` — the SAME harness/metrics as the classical, hybrid and LLM
methods, so the number drops straight into the UC1 comparison table.

**Requires a CUDA GPU** (Colab T4 or the Windows box — not Apple MPS). Install:
```
pip install recbole    # may pin numpy/pandas; use a fresh env if it clashes
```

**Split:** leave-one-out by time, matching `leave_last_n_out(sample, n=1)` — each
user's last interaction is the test target, the second-to-last is validation.

**Scale caveat:** ~12.6M interactions / 50k users. Fine on a T4; the 4 GB Quadro may
OOM → subsample users if needed. RecBole is **GPU (PyTorch)**, not TPU.

In [ ]:
# Keep Kaggle/Colab's native NumPy 2.x — do NOT downgrade numpy. Downgrading breaks the
# binary ABI of every preinstalled package built against NumPy 2 ("numpy.dtype size
# changed"). Just install RecBole; the next cell patches the 3 NumPy-2.0-removed aliases
# its compat shim needs. (If you previously downgraded numpy: Run -> Factory reset first.)
# kmeans-pytorch is an optional RecBole dep that `pip install recbole` doesn't pull in.
!pip install -q recbole kmeans-pytorch

In [ ]:
import os
import numpy as np
import pandas as pd

# RecBole's compatibility_settings() references aliases NumPy 2.0 removed (np.float_,
# np.complex_, np.unicode_), crashing on Kaggle/Colab's native NumPy 2.x. Re-add them so
# RecBole runs WITHOUT downgrading numpy (a downgrade ABI-breaks pandas/scipy). RecBole
# then restores np.float / np.int / etc. itself for the rest of its code.
for _old, _new in [("float_", "float64"), ("complex_", "complex128"), ("unicode_", "str_")]:
    if not hasattr(np, _old):
        setattr(np, _old, getattr(np, _new))

from book_recsys.data.splits import leave_last_n_out
from book_recsys.data.schema import USER, BOOK
from book_recsys.eval.harness import build_relevance, evaluate_predictions
from book_recsys.recbole_adapter.atomic import write_inter_file
from book_recsys.recbole_adapter.export import recbole_predictions

In [ ]:
import glob

def find_data(name):
    """Locate a data file locally (artifacts/) or on Kaggle (/kaggle/input/<dataset>/)."""
    for path in [f"artifacts/{name}", *glob.glob(f"/kaggle/input/**/{name}", recursive=True)]:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"{name} not in artifacts/ or /kaggle/input/** — attach the dataset")

sample = pd.read_parquet(find_data("sample.parquet"))
print(f"loaded {sample['user_id'].nunique()} users, {len(sample):,} interactions")

# --- Memory control for SASRec sequence augmentation (one sub-sequence per interaction) ---
# 1) Subsample users. A RANDOM subset is an unbiased estimate of the SAME NDCG@10, so it
#    stays comparable to the other methods. With the history cap below you can afford more.
N_USERS = 20000
keep = sample["user_id"].drop_duplicates().sample(N_USERS, random_state=42)
sample = sample[sample["user_id"].isin(keep)]

# 2) Cap each user's history to the most recent MAX_HIST interactions. Power users (max
#    ~67k) otherwise generate ~67k sub-sequences each, dominating RAM and the training
#    signal. SASRec only attends to the last MAX_ITEM_LIST_LENGTH (50) items anyway, so
#    trimming older history is principled, not just a hack. tail() keeps the recent items,
#    so each user's held-out LAST interaction is preserved.
MAX_HIST = 100
sample = (sample.sort_values(["user_id", "timestamp"])
                .groupby("user_id", sort=False).tail(MAX_HIST).reset_index(drop=True))
print(f"after subsample + history cap: {sample['user_id'].nunique()} users, "
      f"{len(sample):,} interactions")

# Our shared holdout = each user's last interaction by time. Classical/LLM methods
# are scored against this same `relevance`, so the SASRec number is comparable.
train, holdout = leave_last_n_out(sample, n=1)
relevance = build_relevance(holdout)

DATASET = "goodreads"
ds_dir = f"recbole_data/{DATASET}"
os.makedirs(ds_dir, exist_ok=True)
# Write the FULL (subsampled, capped) data; RecBole reproduces leave-one-out itself (LS
# split in the next cell), so its internal test == our `holdout`. Do NOT pre-split here.
write_inter_file(sample, f"{ds_dir}/{DATASET}.inter")

In [ ]:
# --- 1. Config ---
import json
from logging import getLogger

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, get_trainer, init_logger, init_seed
from recbole.utils.case_study import full_sort_topk

MODEL = "SASRec"
config = Config(model=MODEL, dataset=DATASET, config_dict={
    "data_path": "recbole_data", "dataset": DATASET,
    "USER_ID_FIELD": "user_id", "ITEM_ID_FIELD": "item_id",
    "RATING_FIELD": "rating", "TIME_FIELD": "timestamp",
    "load_col": {"inter": ["user_id", "item_id", "rating", "timestamp"]},
    # Leave-one-out by time == our leave_last_n_out(n=1): last item -> test,
    # second-to-last -> validation. `mode: full` ranks against the WHOLE catalog
    # (matches our harness — NOT sampled negatives, which would inflate metrics).
    "eval_args": {"split": {"LS": "valid_and_test"}, "order": "TO",
                  "group_by": "user", "mode": "full"},
    # Memory lever: shorter sequences = fewer/smaller padded rows. Drop to 20 if RAM-tight.
    "MAX_ITEM_LIST_LENGTH": 50,
    # SASRec uses CE (softmax over all items) loss -> negative sampling must be OFF.
    "train_neg_sample_args": None,
    "epochs": 50, "topk": 10,
    # Speed knobs for the Kaggle/Colab GPU run:
    "train_batch_size": 4096,     # bigger batches = faster GPU throughput
    "eval_step": 5,               # validate every 5 epochs — full-sort over 468k items is slow
    "stopping_step": 3,           # early-stop patience, in eval steps
    "metrics": ["NDCG", "Recall", "MRR"], "valid_metric": "NDCG@10",
    # Cache the processed dataset/dataloaders -> re-runs reload in seconds instead of
    # rebuilding (NB: doesn't lower the FIRST build's peak RAM — the user subsample does).
    "save_dataset": True, "save_dataloaders": True,
    # Checkpoints + logging (a dropped Kaggle session loses an untracked run):
    "checkpoint_dir": "saved",    # best model written here; under cwd (=/kaggle/working/...)
    "show_progress": True,        # per-epoch tqdm bars
    "seed": 42, "reproducibility": True,
})
init_seed(config["seed"], config["reproducibility"])
init_logger(config)               # streams per-epoch loss/valid; writes log/<run>.log
logger = getLogger()

In [ ]:
# --- 2. Build dataset + leave-one-out split (logs user/item/interaction counts) ---
dataset = create_dataset(config)
logger.info(dataset)
train_data, valid_data, test_data = data_preparation(config, dataset)

In [ ]:
# --- 3. Train (best checkpoint saved to saved/ as NDCG@10 improves) ---
model = get_model(config["model"])(config, train_data.dataset).to(config["device"])
trainer = get_trainer(config["MODEL_TYPE"], config["model"])(config, model)
best_score, best_result = trainer.fit(train_data, valid_data, saved=True, show_progress=True)
logger.info("[%s] best valid %s -> checkpoint %s", MODEL, best_result, trainer.saved_model_file)

In [ ]:
# --- 4. Export top-K per test user, map RecBole internal ids -> our ids ---
internal_users = list(range(1, dataset.user_num))  # skip [PAD]=0
_, topk_iid = full_sort_topk(internal_users, model, test_data, k=10, device=config["device"])
uid2token = {i: dataset.id2token(dataset.uid_field, i) for i in internal_users}
iid2token = {int(i): dataset.id2token(dataset.iid_field, int(i))
             for i in topk_iid.reshape(-1).unique().tolist()}
preds = recbole_predictions(internal_users, topk_iid.tolist(), uid2token, iid2token)

In [ ]:
# --- 5. Sanity-check the riskiest seam: internal ids map back to ids we recognize ---
u0 = internal_users[0]
print("RecBole user", u0, "-> our user_id", uid2token[u0])
print("its top-10 book_ids:", preds[uid2token[u0]][:10])
print("# users predicted:", len(preds), "| # users in relevance:", len(relevance))

In [ ]:
# --- 6. Score through our shared harness + persist ---
results = {MODEL: evaluate_predictions(preds, relevance, k=10)}
with open("sasrec_results.json", "w") as fh:   # survives the Kaggle session
    json.dump(results, fh, indent=2)
pd.DataFrame(results).T

## Results, checkpoints & logging

The DataFrame above (recall@10 / ndcg@10 / mrr) feeds `reports/model_report.md`,
in the same UC1 table as svd / hybrid / content / llm. It's also written to
`sasrec_results.json`.

- **Checkpoints:** `saved=True` writes the best model (by NDCG@10) to `saved/` as it
  improves — survives a crash/timeout. Resume a dropped run with
  `trainer.resume_checkpoint("saved/<file>.pth")` before `trainer.fit`.
- **Progress:** `show_progress=True` shows per-epoch tqdm bars; `init_logger` streams
  per-epoch train-loss / valid-NDCG to stdout and writes `log/<run>.log`.
- **Kaggle persistence:** files under `/kaggle/working` (`saved/`, `sasrec_results.json`,
  `log/`) only persist when you **Save Version (Commit)**. For a multi-hour run use
  *Save & Run All (Commit)* so it trains headless and keeps the output even if your
  browser disconnects.

**Aligned:** RecBole's internal leave-one-out test == our `holdout`, and `mode: "full"`
ranks the whole catalog — SASRec is measured on the exact ruler as every other method.
**Sanity-check once:** confirm `uid2token` / `iid2token` map RecBole ids back to
user/book ids you recognize (the riskiest seam; adapter unit tests cover the mapping).